In [ ]:
TRAIN = '\nimport os, sys, math, copy, time, json, glob\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport numpy as np, pandas as pd, torch, cv2\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom sklearn.metrics import roc_auc_score\n\nA1, A2, SRC, TEACH_W, FOLD, EPOCHS = sys.argv[1], sys.argv[2], sys.argv[3], float(sys.argv[4]), int(sys.argv[5]), int(sys.argv[6])\nMODE, BACKBONE, R1 = sys.argv[7], sys.argv[8], [d for d in sys.argv[9].split(",") if d]\nsys.path.insert(0, SRC)\nfrom knee import LABELS, N_SLOTS, KneeNet, window_centres\n\nRES, K, K_EVAL, BS = 256, 8, 12, 2\nLR, LR_HEAD, WD, EMA = (2e-4 if "efficientnet" in BACKBONE else 1e-4), 5e-4, 0.02, 0.998\nddp = int(os.environ.get("WORLD_SIZE", "1")) > 1\nif ddp:\n    dist.init_process_group("nccl")\nrank = dist.get_rank() if ddp else 0\nworld = dist.get_world_size() if ddp else 1\ntorch.cuda.set_device(rank)\ntorch.manual_seed(FOLD * 100 + rank)\nnp.random.seed(FOLD * 100 + rank)\nsay = (lambda m: print(m, flush=True)) if rank == 0 else (lambda m: None)\n\nlab = pd.read_csv(f"{A1}/labels_cons.csv", index_col=0)\nmeta = pd.read_csv(f"{A1}/cache256_meta.csv").set_index("StudyInstanceUID")\nok = set(meta.index[meta.n_slots > 0])\nteach = pd.read_csv(f"{A2}/teacher_coat384_train.csv").set_index("StudyInstanceUID")[LABELS]\ntfail = (teach.values == 0.5).all(1)\nteach = teach[~tfail]\ncons = lab[LABELS]\ntarget = cons.copy()\ncommon = target.index.intersection(teach.index)\ntarget.loc[common] = (1 - TEACH_W) * cons.loc[common] + TEACH_W * teach.loc[common]\nif MODE == "r2":\n    # round 2: image pseudo-label = mean(teacher, clean 5-fold student OOF); LLM share per finding follows how\n    # well the report labels match the gold radiologist labels (strong: ligaments/medial meniscus; weak: effusion,\n    # synovitis, contusion, fracture, lateral OA)\n    oof = pd.concat([pd.read_csv(p, index_col=0) for d in R1 for p in glob.glob(f"{d}/oof_student_f*.csv")])\n    oof = oof[oof.set == "val"][LABELS]\n    oof = oof[~oof.index.duplicated()]\n    say(f"r2: OOF studies {len(oof)}")\n    LLM_SHARE = {"ACL": 0.6, "MCL": 0.6, "Medial Meniscus": 0.5, "Lateral Meniscus": 0.4, "Medial OA": 0.4,\n                 "Lateral OA": 0.2, "PF OA": 0.4, "Effusion": 0.2, "Synovitis": 0.2, "Baker\'s": 0.4,\n                 "Contusion": 0.2, "Fracture": 0.25}\n    img = teach.reindex(cons.index)\n    both = img.index.intersection(oof.index)\n    img.loc[both] = 0.5 * img.loc[both] + 0.5 * oof.loc[both]\n    img = img.fillna(cons)\n    for c in LABELS:\n        target[c] = LLM_SHARE[c] * cons[c] + (1 - LLM_SHARE[c]) * img[c]\ngold = lab[(lab.is_gold == 1) & lab.index.isin(ok)]\nweak = lab[(lab.is_gold == 0) & lab.index.isin(ok)]\ntr = weak[weak.fold != FOLD]\nva = weak[weak.fold == FOLD]\nsay(f"train {len(tr)} val {len(va)} gold {len(gold)} teacher-covered {len(common)} teach_w {TEACH_W}")\nCACHE = f"{A1}/cache256"\n\n\nclass DS(torch.utils.data.Dataset):\n    def __init__(self, ids, y, k, train):\n        self.ids, self.y, self.k, self.train = list(ids), y, k, train\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, i):\n        st = self.ids[i]\n        x = np.zeros((N_SLOTS, self.k, 3, RES, RES), np.uint8)\n        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)\n        with np.load(f"{CACHE}/{st}.npz") as z:\n            for s in range(N_SLOTS):\n                if f"s{s}_o" not in z.files:\n                    continue\n                data, off = z[f"s{s}_d"], z[f"s{s}_o"]\n                n = len(off) - 1\n                c = window_centres(n, self.k, self.train)\n                dec = {}\n                for j in sorted({int(t) for jc in c for t in np.clip([jc - 1, jc, jc + 1], 0, n - 1)}):\n                    dec[j] = cv2.imdecode(data[off[j]:off[j + 1]], cv2.IMREAD_GRAYSCALE)\n                x[s] = np.stack([np.stack([dec[int(t)] for t in np.clip([jc - 1, jc, jc + 1], 0, n - 1)]) for jc in c])\n                mask[s], pos[s] = True, c / max(n - 1, 1)\n        y = self.y[i] if self.y is not None else np.zeros(len(LABELS), np.float32)\n        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))\n\n\ntrain_ds = DS(tr.index, target.loc[tr.index, LABELS].values.astype(np.float32), K, True)\nsampler = torch.utils.data.distributed.DistributedSampler(train_ds, shuffle=True, drop_last=True) if ddp else None\ntrain_dl = torch.utils.data.DataLoader(train_ds, batch_size=BS, shuffle=sampler is None, sampler=sampler, drop_last=True,\n                                       num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=3)\n\nmodel = KneeNet(BACKBONE).cuda().to(memory_format=torch.channels_last)\nema = copy.deepcopy(model).eval()\nfor p in ema.parameters():\n    p.requires_grad_(False)\nenc = list(model.enc.parameters())\nhead = [p for n, p in model.named_parameters() if not n.startswith("enc.")]\nopt = torch.optim.AdamW([{"params": enc, "lr": LR}, {"params": head, "lr": LR_HEAD}], weight_decay=WD)\nnet = torch.nn.parallel.DistributedDataParallel(model, device_ids=[rank]) if ddp else model\nsteps = EPOCHS * len(train_dl)\nwarm = len(train_dl)\nsched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))\nscaler = torch.cuda.amp.GradScaler()\n\n\n@torch.no_grad()\ndef predict(m, ids):\n    dl = torch.utils.data.DataLoader(DS(ids, None, K_EVAL, False), batch_size=4, num_workers=2, pin_memory=True)\n    out = []\n    for x, mask, pos, _ in dl:\n        with torch.autocast("cuda", dtype=torch.float16):\n            out.append(m(x.cuda(non_blocking=True), mask.cuda(), pos.cuda(), res=RES).float().sigmoid().cpu())\n    return torch.cat(out).numpy()\n\n\ndef macro(y, p):\n    a = [roc_auc_score(y[:, i], p[:, i]) for i in range(y.shape[1]) if 0 < y[:, i].sum() < len(y)]\n    return float(np.mean(a)), a\n\n\ngold_y = gold[[f"gold_{c}" for c in LABELS]].values.astype(int)\nt_gold = teach.reindex(gold.index)[LABELS].fillna(0.5).values\nif rank == 0:\n    say(f"teacher gold58 macro {macro(gold_y, t_gold)[0]:.4f}")\nstep, t0 = 0, time.time()\nfor ep in range(EPOCHS):\n    if sampler is not None:\n        sampler.set_epoch(ep)\n    net.train()\n    tot, n, te = 0.0, 0, time.time()\n    for x, mask, pos, y in train_dl:\n        x, mask, pos, y = x.cuda(non_blocking=True), mask.cuda(), pos.cuda(), y.cuda()\n        with torch.autocast("cuda", dtype=torch.float16):\n            logit = net(x, mask, pos, res=RES, train=True)\n        loss = F.binary_cross_entropy_with_logits(logit.float(), y)\n        opt.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)\n        scaler.step(opt)\n        scaler.update()\n        sched.step()\n        step += 1\n        d = min(EMA, (1 + step) / (10 + step))\n        with torch.no_grad():\n            for pe, pm in zip(ema.parameters(), model.parameters()):\n                pe.lerp_(pm, 1 - d)\n            for be, bm in zip(ema.buffers(), model.buffers()):\n                be.copy_(bm)\n        tot += loss.item()\n        n += 1\n        if rank == 0 and n % 100 == 0:\n            print(f"  ep {ep} {n}/{len(train_dl)} loss {tot/n:.4f} {(time.time()-te)/n:.2f}s/it", flush=True)\n    if rank == 0:\n        pg = predict(ema, gold.index)\n        g, ge = macro(gold_y, pg)\n        rk = lambda a: pd.DataFrame(a).rank(pct=True).values\n        gb = macro(gold_y, 0.5 * rk(pg) + 0.5 * rk(t_gold))[0]\n        say(f"ep {ep:2d} loss {tot/n:.4f} {time.time()-te:.0f}s | gold student {g:.4f} | gold rank(student+teacher) {gb:.4f} | total {time.time()-t0:.0f}s")\n        torch.save({"model": ema.state_dict(), "args": {"backbone": BACKBONE, "res": RES, "k_eval": K_EVAL, "mode": MODE,\n                    "fold": FOLD, "teach_w": TEACH_W, "epoch": ep}}, (f"/kaggle/working/student_f{FOLD}.pt" if MODE == "r1" else f"/kaggle/working/student_{MODE}_{BACKBONE}_f{FOLD}.pt"))\n    if ddp:\n        dist.barrier()\n\nif rank == 0:\n    pv = predict(ema, va.index)\n    pg = predict(ema, gold.index)\n    vy = (lab.loc[va.index, LABELS].values > 0.5).astype(int)\n    v, ve = macro(vy, pv)\n    g, ge = macro(gold_y, pg)\n    say(f"FINAL fold {FOLD} | val(consensus) {v:.4f} | gold58 {g:.4f}")\n    say("gold per class: " + " ".join(f"{c[:6]}={a:.3f}" for c, a in zip(LABELS, ge)))\n    df = pd.concat([pd.DataFrame(pv, index=va.index, columns=LABELS).assign(set="val"),\n                    pd.DataFrame(pg, index=gold.index, columns=LABELS).assign(set="gold")])\n    df.to_csv(f"/kaggle/working/oof_student_f{FOLD}.csv" if MODE == "r1" else f"/kaggle/working/oof_{MODE}_{BACKBONE}_f{FOLD}.csv")\nif ddp:\n    dist.destroy_process_group()\n'
FOLDS = [0,1,2,3,4]
MODE = 'r2'
BACKBONE = 'convnext_tiny'

import os, sys, glob, subprocess, shutil, time
def kmount(slug, required):
    for p in glob.glob(f"/kaggle/input/notebooks/*/{slug}") + glob.glob(f"/kaggle/input/{slug}") + glob.glob(f"/kaggle/input/*/{slug}"):
        if os.path.exists(os.path.join(p, required)):
            return p
    raise FileNotFoundError(slug)
A1 = kmount("rsna-knee-a1-train-cache-labels", "labels_cons.csv")
A2 = kmount("rsna-knee-a2-teacher-train-preds", "teacher_coat384_train.csv")
GP = next(p for p in ("/kaggle/input/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader",
                      "/kaggle/input/rsna-knee-2-5d-convnext-reader") if os.path.exists(p + "/knee.py"))
SRC = "/kaggle/working/src"
os.makedirs(SRC, exist_ok=True)
shutil.copy(GP + "/knee.py", SRC)
open(SRC + "/train_student.py", "w").write(TRAIN)
R1 = ",".join(kmount(s, "student_f0.pt" if s.endswith("f0") else "student_f1.pt")
               for s in ("rsna-knee-b-student-f0", "rsna-knee-b-student-f1234")) if MODE == "r2" else ""
import timm
timm.create_model(BACKBONE, pretrained=True)  # cache the ImageNet weights once before the workers start
for fold in FOLDS:
    t0 = time.time()
    rc = subprocess.run(["torchrun", "--standalone", "--nproc_per_node", "2", SRC + "/train_student.py",
                         A1, A2, SRC, "0.6", str(fold), "12", MODE, BACKBONE, R1]).returncode
    print("fold", fold, "rc", rc, f"{time.time()-t0:.0f}s", flush=True)
    assert rc == 0
